[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ElenaVillano/prope-programacion/blob/main/materiales/m07_agrupacion.ipynb)

# Propedéutico de Programación para el Análisis de datos

## EGobiernoyTP

**Verano 2026**

### Material 6: Exploración de bases de datos: intro pandas





# Agrupar y combinar bases de datos con pandas

En este notebook trabajaremos dos habilidades fundamentales para el análisis de datos:

1. **Agrupar información** con `groupby()` y `agg()`.
2. **Combinar bases de datos** con `merge()` y distintos tipos de joins.

Usaremos como base principal el DataFrame `df`, que contiene variables como `state`, `group`, `subgroup`, `phase` y `value`.


In [1]:
# Librerías 
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/ElenaVillano/prope-programacion/refs/heads/main/data/datos.csv')

## Preparación

Este notebook asume que `df` ya está cargado en memoria y que los nombres de sus columnas fueron estandarizados en minúsculas y con guiones bajos.


In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.head()


,indicator,group,state,subgroup,phase,time_period,time_period_label,time_period_start_date,time_period_end_date,value,lowci,highci,confidence_interval,quartile_range,suppression_flag
0,"Received Counseling or Therapy, Last 4 Weeks",By Sex,United States,Male,2,15,"Sep 16 - Sep 28, 2020",09/16/2020,09/28/2020,6.9,6.5,7.3,6.5 - 7.3,NaN,NaN
1,"Received Counseling or Therapy, Last 4 Weeks",By Sex,United States,Female,2,15,"Sep 16 - Sep 28, 2020",09/16/2020,09/28/2020,11.0,10.4,11.6,10.4 - 11.6,NaN,NaN
2,Needed Counseling or Therapy But Did Not Get I...,By Sex,United States,Female,-1,1,"Dec 22, 2020 - Jan 5, 2021",12/22/2020,01/05/2021,NaN,NaN,NaN,NaN,NaN,NaN
3,Took Prescription Medication for Mental Health...,By Age,United States,50 - 59 years,-1,1,"Mar 30 - Apr 13, 2021",03/30/2021,04/13/2021,NaN,NaN,NaN,NaN,NaN,NaN
4,Took Prescription Medication for Mental Health...,By Age,United States,60 - 69 years,-1,1,"Mar 30 - Apr 13, 2021",03/30/2021,04/13/2021,NaN,NaN,NaN,NaN,NaN,NaN


# 1. Agrupar información

Muchas veces no queremos analizar cada observación individualmente, sino **resumir información para distintos grupos**.

Por ejemplo:

> ¿Cuál es el valor promedio para cada estado?


In [4]:
df[["state", "value"]].head()


,state,value
0,United States,6.9
1,United States,11.0
2,United States,NaN
3,United States,NaN
4,United States,NaN


## `groupby()`

`groupby()` divide las observaciones de acuerdo con los valores de una variable.

Por sí solo, `groupby()` todavía no calcula nada: después debemos indicar qué operación queremos realizar dentro de cada grupo.


In [5]:
df.groupby("state")


### Promedio por estado


In [6]:
df.groupby("state")["value"].mean()


state
Alabama                 18.059848
Alaska                  15.571212
Arizona                 15.950758
Arkansas                18.710606
California              15.145455
Colorado                18.284848
Connecticut             17.377273
Delaware                16.509091
District of Columbia    19.846212
Florida                 15.122727
Georgia                 16.569697
Hawaii                  11.590840
Idaho                   17.876515
Illinois                16.637121
Indiana                 18.278030
Iowa                    18.037121
Kansas                  17.917424
Kentucky                19.399242
Louisiana               18.418182
Maine                   19.362121
Maryland                16.956818
Massachusetts           19.316667
Michigan                17.634091
Minnesota               18.082576
Mississippi             16.712121
Missouri                18.358333
Montana                 16.959848
Nebraska                17.317424
Nevada                  14.468939
New Hamp

Podemos guardar el resultado en una nueva variable:


In [7]:
promedio_estado = (
    df
    .groupby("state")["value"]
    .mean()
)

promedio_estado.head()


state
Alabama       18.059848
Alaska        15.571212
Arizona       15.950758
Arkansas      18.710606
California    15.145455
Name: value, dtype: float64

### ¿Qué pasó con `state`?

Después de agrupar, `state` se convierte en el índice del resultado.


In [8]:
promedio_estado.index


Index(['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado',
       'Connecticut', 'Delaware', 'District of Columbia', 'Florida', 'Georgia',
       'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky',
       'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan',
       'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada',
       'New Hampshire', 'New Jersey', 'New Mexico', 'New York',
       'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon',
       'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota',
       'Tennessee', 'Texas', 'United States', 'Utah', 'Vermont', 'Virginia',
       'Washington', 'West Virginia', 'Wisconsin', 'Wyoming'],
      dtype='str', name='state')

Podemos regresar el índice a una columna con `reset_index()`:


In [9]:
promedio_estado = (
    df
    .groupby("state")["value"]
    .mean()
    .reset_index()
)

promedio_estado.head()


,state,value
0,Alabama,18.059848
1,Alaska,15.571212
2,Arizona,15.950758
3,Arkansas,18.710606
4,California,15.145455


## Otras operaciones de resumen

Además del promedio, podemos calcular otros estadísticos por grupo.


In [10]:
df.groupby("state")["value"].count()


state
Alabama                  132
Alaska                   132
Arizona                  132
Arkansas                 132
California               132
Colorado                 132
Connecticut              132
Delaware                 132
District of Columbia     132
Florida                  132
Georgia                  132
Hawaii                   131
Idaho                    132
Illinois                 132
Indiana                  132
Iowa                     132
Kansas                   132
Kentucky                 132
Louisiana                132
Maine                    132
Maryland                 132
Massachusetts            132
Michigan                 132
Minnesota                132
Mississippi              132
Missouri                 132
Montana                  132
Nebraska                 132
Nevada                   132
New Hampshire            132
New Jersey               132
New Mexico               132
New York                 132
North Carolina           132
North Da

In [11]:
df.groupby("state")["value"].min()


state
Alabama                  6.2
Alaska                   7.5
Arizona                  6.7
Arkansas                 5.5
California               8.8
Colorado                 9.5
Connecticut              5.7
Delaware                 4.6
District of Columbia     8.7
Florida                  6.5
Georgia                  6.6
Hawaii                   3.3
Idaho                    7.4
Illinois                 7.6
Indiana                  6.7
Iowa                     6.1
Kansas                   6.8
Kentucky                 5.8
Louisiana                6.8
Maine                    6.4
Maryland                 7.6
Massachusetts            7.9
Michigan                 7.3
Minnesota                7.3
Mississippi              4.1
Missouri                 5.6
Montana                  7.0
Nebraska                 6.2
Nevada                   6.3
New Hampshire            6.8
New Jersey               6.0
New Mexico               6.5
New York                 6.6
North Carolina           7.2
North Da

In [12]:
df.groupby("state")["value"].max()


state
Alabama                 33.0
Alaska                  30.6
Arizona                 27.9
Arkansas                35.5
California              25.9
Colorado                32.0
Connecticut             30.8
Delaware                31.8
District of Columbia    35.3
Florida                 26.2
Georgia                 28.3
Hawaii                  23.0
Idaho                   31.5
Illinois                27.4
Indiana                 31.5
Iowa                    33.6
Kansas                  33.6
Kentucky                36.2
Louisiana               36.3
Maine                   34.8
Maryland                30.7
Massachusetts           33.0
Michigan                30.9
Minnesota               32.9
Mississippi             30.8
Missouri                32.9
Montana                 32.0
Nebraska                32.6
Nevada                  24.9
New Hampshire           33.4
New Jersey              26.5
New Mexico              28.7
New York                28.3
North Carolina          32.8
North Da

In [13]:
df.groupby("state")["value"].median()


state
Alabama                 17.00
Alaska                  14.90
Arizona                 16.50
Arkansas                17.70
California              14.65
Colorado                16.95
Connecticut             17.05
Delaware                16.00
District of Columbia    19.45
Florida                 14.60
Georgia                 17.20
Hawaii                  11.20
Idaho                   17.85
Illinois                15.95
Indiana                 18.05
Iowa                    17.75
Kansas                  17.55
Kentucky                20.55
Louisiana               17.90
Maine                   18.45
Maryland                16.35
Massachusetts           19.20
Michigan                16.70
Minnesota               16.15
Mississippi             17.00
Missouri                19.70
Montana                 16.00
Nebraska                16.55
Nevada                  15.25
New Hampshire           16.55
New Jersey              14.30
New Mexico              15.50
New York                15.00
Nort

## `agg()`

Cuando queremos calcular **varios estadísticos al mismo tiempo**, podemos utilizar `agg()`.


In [14]:
resumen_estado = (
    df
    .groupby("state")["value"]
    .agg(["count", "mean", "median", "min", "max"])
)

resumen_estado.head()


,count,mean,median,min,max
state,,,,,
Alabama,132,18.059848,17.00,6.2,33.0
Alaska,132,15.571212,14.90,7.5,30.6
Arizona,132,15.950758,16.50,6.7,27.9
Arkansas,132,18.710606,17.70,5.5,35.5
California,132,15.145455,14.65,8.8,25.9


Podemos regresar `state` a una columna:


In [15]:
resumen_estado = resumen_estado.reset_index()
resumen_estado.head()


,state,count,mean,median,min,max
0,Alabama,132,18.059848,17.00,6.2,33.0
1,Alaska,132,15.571212,14.90,7.5,30.6
2,Arizona,132,15.950758,16.50,6.7,27.9
3,Arkansas,132,18.710606,17.70,5.5,35.5
4,California,132,15.145455,14.65,8.8,25.9


### Nombrar los resultados

También podemos decidir cómo queremos llamar a cada nueva columna.

La sintaxis general es:

```python
nombre_nuevo = ("variable", "operación")
```


In [16]:
resumen_estado = (
    df
    .groupby("state")
    .agg(
        observaciones=("value", "count"),
        promedio=("value", "mean"),
        mediana=("value", "median"),
        minimo=("value", "min"),
        maximo=("value", "max")
    )
    .reset_index()
)

resumen_estado.head()


,state,observaciones,promedio,mediana,minimo,maximo
0,Alabama,132,18.059848,17.00,6.2,33.0
1,Alaska,132,15.571212,14.90,7.5,30.6
2,Arizona,132,15.950758,16.50,6.7,27.9
3,Arkansas,132,18.710606,17.70,5.5,35.5
4,California,132,15.145455,14.65,8.8,25.9


## Agrupar por más de una variable

También podemos definir grupos a partir de **más de una característica**.

Por ejemplo, calcular el promedio para cada combinación de `state` y `group`.


In [17]:
df.groupby(
    ["state", "group"]
)["value"].mean()


state                 group                                        
Alabama               By State                                         18.059848
Alaska                By State                                         15.571212
Arizona               By State                                         15.950758
Arkansas              By State                                         18.710606
California            By State                                         15.145455
Colorado              By State                                         18.284848
Connecticut           By State                                         17.377273
Delaware              By State                                         16.509091
District of Columbia  By State                                         19.846212
Florida               By State                                         15.122727
Georgia               By State                                         16.569697
Hawaii                By State           

Podemos producir directamente un DataFrame resumido:


In [18]:
promedio_estado_grupo = (
    df
    .groupby(["state", "group"])
    .agg(
        promedio=("value", "mean"),
        observaciones=("value", "count")
    )
    .reset_index()
)

promedio_estado_grupo.head(10)


,state,group,promedio,observaciones
0,Alabama,By State,18.059848,132
1,Alaska,By State,15.571212,132
2,Arizona,By State,15.950758,132
3,Arkansas,By State,18.710606,132
4,California,By State,15.145455,132
5,Colorado,By State,18.284848,132
6,Connecticut,By State,17.377273,132
7,Delaware,By State,16.509091,132
8,District of Columbia,By State,19.846212,132
9,Florida,By State,15.122727,132


También podemos agregar una tercera variable de agrupación:


In [19]:
promedio_estado_grupo_fase = (
    df
    .groupby(["state", "group", "phase"])
    .agg(
        promedio=("value", "mean"),
        observaciones=("value", "count")
    )
    .reset_index()
)

promedio_estado_grupo_fase.head()


,state,group,phase,promedio,observaciones
0,Alabama,By State,2,16.585000,20
1,Alabama,By State,3 (Jan 6 – Mar 29),17.120833,24
2,Alabama,By State,3 (Oct 28 – Dec 21),18.250000,16
3,Alabama,By State,3.1,17.554167,24
4,Alabama,By State,3.2,19.600000,24


## Agrupar y ordenar

Podemos combinar varias operaciones. Por ejemplo:

> ¿Qué estados tienen el promedio más alto?


In [20]:
(
    df
    .groupby("state")
    .agg(
        promedio=("value", "mean")
    )
    .reset_index()
    .sort_values(
        "promedio",
        ascending=False
    )
    .head(10)
)


,state,promedio
45,Utah,20.082576
37,Oregon,19.865909
8,District of Columbia,19.846212
49,West Virginia,19.735606
46,Vermont,19.661240
39,Rhode Island,19.604545
17,Kentucky,19.399242
19,Maine,19.362121
21,Massachusetts,19.316667
36,Oklahoma,18.909091


# 2. Combinar bases de datos

En la práctica, la información rara vez está contenida en una sola tabla.

Por ejemplo, podemos tener una tabla con observaciones y otra con información adicional sobre cada estado.

La variable que permite relacionarlas se conoce como **llave** o **identificador**.


## Crear una segunda tabla

Construiremos una tabla pequeña con la región de algunos estados para practicar los joins.


In [21]:
import pandas as pd

regiones = pd.DataFrame({
    "state": [
        "Vermont",
        "Wyoming",
        "Hawaii",
        "North Dakota",
        "California"
    ],
    "region": [
        "Northeast",
        "West",
        "West",
        "Midwest",
        "West"
    ]
})

regiones


,state,region
0,Vermont,Northeast
1,Wyoming,West
2,Hawaii,West
3,North Dakota,Midwest
4,California,West


Ahora tenemos dos tablas que comparten la variable `state`:


In [22]:
df[["state", "value"]].head()


,state,value
0,United States,6.9
1,United States,11.0
2,United States,NaN
3,United States,NaN
4,United States,NaN


In [23]:
regiones


,state,region
0,Vermont,Northeast
1,Wyoming,West
2,Hawaii,West
3,North Dakota,Midwest
4,California,West


## Revisar la llave antes de combinar

Antes de hacer un `merge()`, conviene revisar si la llave contiene valores faltantes o duplicados.


In [24]:
regiones["state"].isna().sum()


np.int64(0)

In [25]:
regiones["state"].duplicated().sum()


np.int64(0)

In [26]:
regiones["state"].unique()


<StringArray>
['Vermont', 'Wyoming', 'Hawaii', 'North Dakota', 'California']
Length: 5, dtype: str

# `merge()`

Podemos combinar ambas tablas utilizando `merge()`.


In [27]:
df.merge(
    regiones,
    on="state"
).head()


,indicator,group,state,subgroup,phase,time_period,time_period_label,time_period_start_date,time_period_end_date,value,lowci,highci,confidence_interval,quartile_range,suppression_flag,region
0,Took Prescription Medication for Mental Health...,By State,California,California,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,15.8,14.4,17.4,14.4 - 17.4,12.2-18.4,NaN,West
1,Took Prescription Medication for Mental Health...,By State,Hawaii,Hawaii,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,12.2,9.8,15.0,9.8 - 15.0,12.2-18.4,NaN,West
2,Took Prescription Medication for Mental Health...,By State,North Dakota,North Dakota,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,19.6,15.8,23.9,15.8 - 23.9,18.5-20.5,NaN,Midwest
3,Took Prescription Medication for Mental Health...,By State,Vermont,Vermont,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,23.2,18.5,28.5,18.5 - 28.5,22.6-26.8,NaN,Northeast
4,Took Prescription Medication for Mental Health...,By State,Wyoming,Wyoming,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,18.7,15.0,22.7,15.0 - 22.7,18.5-20.5,NaN,West


Una forma equivalente es usar `pd.merge()`:


In [28]:
pd.merge(
    df,
    regiones,
    on="state"
).head()


,indicator,group,state,subgroup,phase,time_period,time_period_label,time_period_start_date,time_period_end_date,value,lowci,highci,confidence_interval,quartile_range,suppression_flag,region
0,Took Prescription Medication for Mental Health...,By State,California,California,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,15.8,14.4,17.4,14.4 - 17.4,12.2-18.4,NaN,West
1,Took Prescription Medication for Mental Health...,By State,Hawaii,Hawaii,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,12.2,9.8,15.0,9.8 - 15.0,12.2-18.4,NaN,West
2,Took Prescription Medication for Mental Health...,By State,North Dakota,North Dakota,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,19.6,15.8,23.9,15.8 - 23.9,18.5-20.5,NaN,Midwest
3,Took Prescription Medication for Mental Health...,By State,Vermont,Vermont,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,23.2,18.5,28.5,18.5 - 28.5,22.6-26.8,NaN,Northeast
4,Took Prescription Medication for Mental Health...,By State,Wyoming,Wyoming,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,18.7,15.0,22.7,15.0 - 22.7,18.5-20.5,NaN,West


# Tipos de joins

El argumento `how` determina qué observaciones queremos conservar.

- `inner`: solo coincidencias.
- `left`: todo lo de la tabla izquierda.
- `right`: todo lo de la tabla derecha.
- `outer`: todo lo de ambas tablas.


## `inner join`

Conserva únicamente las observaciones que aparecen en **ambas tablas**.


In [29]:
df_inner = df.merge(
    regiones,
    on="state",
    how="inner"
)

df_inner.head()


,indicator,group,state,subgroup,phase,time_period,time_period_label,time_period_start_date,time_period_end_date,value,lowci,highci,confidence_interval,quartile_range,suppression_flag,region
0,Took Prescription Medication for Mental Health...,By State,California,California,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,15.8,14.4,17.4,14.4 - 17.4,12.2-18.4,NaN,West
1,Took Prescription Medication for Mental Health...,By State,Hawaii,Hawaii,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,12.2,9.8,15.0,9.8 - 15.0,12.2-18.4,NaN,West
2,Took Prescription Medication for Mental Health...,By State,North Dakota,North Dakota,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,19.6,15.8,23.9,15.8 - 23.9,18.5-20.5,NaN,Midwest
3,Took Prescription Medication for Mental Health...,By State,Vermont,Vermont,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,23.2,18.5,28.5,18.5 - 28.5,22.6-26.8,NaN,Northeast
4,Took Prescription Medication for Mental Health...,By State,Wyoming,Wyoming,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,18.7,15.0,22.7,15.0 - 22.7,18.5-20.5,NaN,West


In [30]:
df.shape, df_inner.shape


((10404, 15), (660, 16))

## `left join`

Conserva **todas las observaciones de la tabla izquierda** y agrega información de la tabla derecha cuando existe una correspondencia.


In [31]:
df_left = df.merge(
    regiones,
    on="state",
    how="left"
)

df_left.head()


,indicator,group,state,subgroup,phase,time_period,time_period_label,time_period_start_date,time_period_end_date,value,lowci,highci,confidence_interval,quartile_range,suppression_flag,region
0,"Received Counseling or Therapy, Last 4 Weeks",By Sex,United States,Male,2,15,"Sep 16 - Sep 28, 2020",09/16/2020,09/28/2020,6.9,6.5,7.3,6.5 - 7.3,NaN,NaN,NaN
1,"Received Counseling or Therapy, Last 4 Weeks",By Sex,United States,Female,2,15,"Sep 16 - Sep 28, 2020",09/16/2020,09/28/2020,11.0,10.4,11.6,10.4 - 11.6,NaN,NaN,NaN
2,Needed Counseling or Therapy But Did Not Get I...,By Sex,United States,Female,-1,1,"Dec 22, 2020 - Jan 5, 2021",12/22/2020,01/05/2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Took Prescription Medication for Mental Health...,By Age,United States,50 - 59 years,-1,1,"Mar 30 - Apr 13, 2021",03/30/2021,04/13/2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Took Prescription Medication for Mental Health...,By Age,United States,60 - 69 years,-1,1,"Mar 30 - Apr 13, 2021",03/30/2021,04/13/2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Si un estado no aparece en `regiones`, la variable `region` tendrá un valor faltante (`NaN`).


In [32]:
df_left[df_left["region"].isna()].head()


,indicator,group,state,subgroup,phase,time_period,time_period_label,time_period_start_date,time_period_end_date,value,lowci,highci,confidence_interval,quartile_range,suppression_flag,region
0,"Received Counseling or Therapy, Last 4 Weeks",By Sex,United States,Male,2,15,"Sep 16 - Sep 28, 2020",09/16/2020,09/28/2020,6.9,6.5,7.3,6.5 - 7.3,NaN,NaN,NaN
1,"Received Counseling or Therapy, Last 4 Weeks",By Sex,United States,Female,2,15,"Sep 16 - Sep 28, 2020",09/16/2020,09/28/2020,11.0,10.4,11.6,10.4 - 11.6,NaN,NaN,NaN
2,Needed Counseling or Therapy But Did Not Get I...,By Sex,United States,Female,-1,1,"Dec 22, 2020 - Jan 5, 2021",12/22/2020,01/05/2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Took Prescription Medication for Mental Health...,By Age,United States,50 - 59 years,-1,1,"Mar 30 - Apr 13, 2021",03/30/2021,04/13/2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Took Prescription Medication for Mental Health...,By Age,United States,60 - 69 years,-1,1,"Mar 30 - Apr 13, 2021",03/30/2021,04/13/2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## `right join`

Conserva todas las observaciones de la tabla de la derecha.


In [33]:
df_right = df.merge(
    regiones,
    on="state",
    how="right"
)

df_right.head()


,indicator,group,state,subgroup,phase,time_period,time_period_label,time_period_start_date,time_period_end_date,value,lowci,highci,confidence_interval,quartile_range,suppression_flag,region
0,Took Prescription Medication for Mental Health...,By State,Vermont,Vermont,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,23.2,18.5,28.5,18.5 - 28.5,22.6-26.8,NaN,Northeast
1,"Received Counseling or Therapy, Last 4 Weeks",By State,Vermont,Vermont,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,11.4,8.8,14.5,8.8 - 14.5,10.1-19.1,NaN,Northeast
2,Took Prescription Medication for Mental Health...,By State,Vermont,Vermont,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,27.8,22.5,33.6,22.5 - 33.6,25.4-28.4,NaN,Northeast
3,Needed Counseling or Therapy But Did Not Get I...,By State,Vermont,Vermont,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,10.5,7.2,14.7,7.2 - 14.7,10.2-12.9,NaN,Northeast
4,Took Prescription Medication for Mental Health...,By State,Vermont,Vermont,2,14,"Sep 2 - Sep 14, 2020",09/02/2020,09/14/2020,22.3,18.9,26.1,18.9 - 26.1,21.3-22.8,NaN,Northeast


## `outer join`

Conserva las observaciones de **ambas tablas**, aunque no tengan correspondencia.


In [34]:
df_outer = df.merge(
    regiones,
    on="state",
    how="outer"
)

df_outer.head()


,indicator,group,state,subgroup,phase,time_period,time_period_label,time_period_start_date,time_period_end_date,value,lowci,highci,confidence_interval,quartile_range,suppression_flag,region
0,Took Prescription Medication for Mental Health...,By State,Alabama,Alabama,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,22.3,19.6,25.1,19.6 - 25.1,20.6-22.5,NaN,NaN
1,"Received Counseling or Therapy, Last 4 Weeks",By State,Alabama,Alabama,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,8.5,6.5,10.9,6.5 - 10.9,7.2-8.7,NaN,NaN
2,Took Prescription Medication for Mental Health...,By State,Alabama,Alabama,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,25.2,21.9,28.6,21.9 - 28.6,23.6-25.3,NaN,NaN
3,Needed Counseling or Therapy But Did Not Get I...,By State,Alabama,Alabama,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,8.4,6.5,10.6,6.5 - 10.6,7.8-9.1,NaN,NaN
4,Took Prescription Medication for Mental Health...,By State,Alabama,Alabama,2,14,"Sep 2 - Sep 14, 2020",09/02/2020,09/14/2020,21.0,17.9,24.3,17.9 - 24.3,18.9-21.2,NaN,NaN


# Identificar observaciones que no hicieron match

Hacer un `merge()` no garantiza que todas las observaciones hayan encontrado correspondencia.

Una primera estrategia es buscar valores faltantes después de un `left join`.


In [35]:
df_left.loc[
    df_left["region"].isna(),
    "state"
].unique()


<StringArray>
[       'United States',              'Alabama',               'Alaska',
              'Arizona',             'Arkansas',             'Colorado',
          'Connecticut',             'Delaware', 'District of Columbia',
              'Florida',              'Georgia',                'Idaho',
             'Illinois',              'Indiana',                 'Iowa',
               'Kansas',             'Kentucky',            'Louisiana',
                'Maine',             'Maryland',        'Massachusetts',
             'Michigan',            'Minnesota',          'Mississippi',
             'Missouri',              'Montana',             'Nebraska',
               'Nevada',        'New Hampshire',           'New Jersey',
           'New Mexico',             'New York',       'North Carolina',
                 'Ohio',             'Oklahoma',               'Oregon',
         'Pennsylvania',         'Rhode Island',       'South Carolina',
         'South Dakota',            '

## `indicator=True`

Pandas puede crear automáticamente una columna que indique de dónde provino cada observación.


In [36]:
df_revision = df.merge(
    regiones,
    on="state",
    how="outer",
    indicator=True
)

df_revision.head()


,indicator,group,state,subgroup,phase,time_period,time_period_label,time_period_start_date,time_period_end_date,value,lowci,highci,confidence_interval,quartile_range,suppression_flag,region,_merge
0,Took Prescription Medication for Mental Health...,By State,Alabama,Alabama,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,22.3,19.6,25.1,19.6 - 25.1,20.6-22.5,NaN,NaN,left_only
1,"Received Counseling or Therapy, Last 4 Weeks",By State,Alabama,Alabama,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,8.5,6.5,10.9,6.5 - 10.9,7.2-8.7,NaN,NaN,left_only
2,Took Prescription Medication for Mental Health...,By State,Alabama,Alabama,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,25.2,21.9,28.6,21.9 - 28.6,23.6-25.3,NaN,NaN,left_only
3,Needed Counseling or Therapy But Did Not Get I...,By State,Alabama,Alabama,2,13,"Aug 19 - Aug 31, 2020",08/19/2020,08/31/2020,8.4,6.5,10.6,6.5 - 10.6,7.8-9.1,NaN,NaN,left_only
4,Took Prescription Medication for Mental Health...,By State,Alabama,Alabama,2,14,"Sep 2 - Sep 14, 2020",09/02/2020,09/14/2020,21.0,17.9,24.3,17.9 - 24.3,18.9-21.2,NaN,NaN,left_only


La columna `_merge` puede tomar tres valores:

- `both`: encontró correspondencia.
- `left_only`: solo estaba en la tabla izquierda.
- `right_only`: solo estaba en la tabla derecha.


In [37]:
df_revision["_merge"].value_counts()


_merge
left_only     9744
both           660
right_only       0
Name: count, dtype: int64

Podemos inspeccionar los estados de `df` que no encontraron correspondencia:


In [38]:
df_revision.loc[
    df_revision["_merge"] == "left_only",
    "state"
].unique()


<StringArray>
[             'Alabama',               'Alaska',              'Arizona',
             'Arkansas',             'Colorado',          'Connecticut',
             'Delaware', 'District of Columbia',              'Florida',
              'Georgia',                'Idaho',             'Illinois',
              'Indiana',                 'Iowa',               'Kansas',
             'Kentucky',            'Louisiana',                'Maine',
             'Maryland',        'Massachusetts',             'Michigan',
            'Minnesota',          'Mississippi',             'Missouri',
              'Montana',             'Nebraska',               'Nevada',
        'New Hampshire',           'New Jersey',           'New Mexico',
             'New York',       'North Carolina',                 'Ohio',
             'Oklahoma',               'Oregon',         'Pennsylvania',
         'Rhode Island',       'South Carolina',         'South Dakota',
            'Tennessee',             

Y los estados que aparecen únicamente en la tabla `regiones`:


In [39]:
df_revision.loc[
    df_revision["_merge"] == "right_only",
    "state"
].unique()


<StringArray>
[]
Length: 0, dtype: str

# Un problema muy común: llaves inconsistentes

Dos valores pueden parecer iguales para una persona, pero no necesariamente para Python.

Por ejemplo, diferencias en mayúsculas, minúsculas o espacios pueden hacer que un `merge()` falle.


In [40]:
regiones_error = pd.DataFrame({
    "state": [
        "Vermont",
        "wyoming",
        "Hawaii ",
        "North Dakota"
    ],
    "region": [
        "Northeast",
        "West",
        "West",
        "Midwest"
    ]
})

regiones_error


,state,region
0,Vermont,Northeast
1,wyoming,West
2,Hawaii,West
3,North Dakota,Midwest


In [41]:
revision = df.merge(
    regiones_error,
    on="state",
    how="left",
    indicator=True
)

revision["_merge"].value_counts()


_merge
left_only     10140
both            264
right_only        0
Name: count, dtype: int64

Podemos revisar qué estados no hicieron match:


In [42]:
revision.loc[
    revision["_merge"] == "left_only",
    "state"
].unique()


<StringArray>
[       'United States',              'Alabama',               'Alaska',
              'Arizona',             'Arkansas',           'California',
             'Colorado',          'Connecticut',             'Delaware',
 'District of Columbia',              'Florida',              'Georgia',
               'Hawaii',                'Idaho',             'Illinois',
              'Indiana',                 'Iowa',               'Kansas',
             'Kentucky',            'Louisiana',                'Maine',
             'Maryland',        'Massachusetts',             'Michigan',
            'Minnesota',          'Mississippi',             'Missouri',
              'Montana',             'Nebraska',               'Nevada',
        'New Hampshire',           'New Jersey',           'New Mexico',
             'New York',       'North Carolina',                 'Ohio',
             'Oklahoma',               'Oregon',         'Pennsylvania',
         'Rhode Island',       'South

Observa que estas comparaciones son falsas:


In [43]:
"Wyoming" == "wyoming"


False

In [44]:
"Hawaii" == "Hawaii "


False

## Limpiar las llaves antes de combinar

Podemos crear versiones estandarizadas de las variables que funcionan como llave.


In [45]:
regiones_error["state_limpio"] = (
    regiones_error["state"]
    .str.strip()
    .str.lower()
)

df["state_limpio"] = (
    df["state"]
    .str.strip()
    .str.lower()
)


Ahora podemos hacer el merge utilizando nombres de llave distintos en cada tabla con `left_on` y `right_on`:


In [46]:
df_merge = df.merge(
    regiones_error,
    left_on="state_limpio",
    right_on="state_limpio",
    how="left",
    indicator=True,
    suffixes=("_df", "_regiones")
)

df_merge["_merge"].value_counts()


_merge
left_only     9876
both           528
right_only       0
Name: count, dtype: int64

# Cardinalidad de las llaves

Antes de combinar bases también debemos preguntarnos:

> ¿La llave identifica de manera única una observación?

En `df` puede haber muchas observaciones por estado, pero en `regiones` esperamos una sola fila por estado. Esto corresponde a una relación **muchos-a-uno**.


In [47]:
regiones["state"].duplicated().sum()


np.int64(0)

Podemos pedirle a pandas que valide esta relación al hacer el merge:


In [48]:
df_merge_validado = df.merge(
    regiones,
    on="state",
    how="left",
    validate="many_to_one"
)

df_merge_validado.head()


,indicator,group,state,subgroup,phase,time_period,time_period_label,time_period_start_date,time_period_end_date,value,lowci,highci,confidence_interval,quartile_range,suppression_flag,state_limpio,region
0,"Received Counseling or Therapy, Last 4 Weeks",By Sex,United States,Male,2,15,"Sep 16 - Sep 28, 2020",09/16/2020,09/28/2020,6.9,6.5,7.3,6.5 - 7.3,NaN,NaN,united states,NaN
1,"Received Counseling or Therapy, Last 4 Weeks",By Sex,United States,Female,2,15,"Sep 16 - Sep 28, 2020",09/16/2020,09/28/2020,11.0,10.4,11.6,10.4 - 11.6,NaN,NaN,united states,NaN
2,Needed Counseling or Therapy But Did Not Get I...,By Sex,United States,Female,-1,1,"Dec 22, 2020 - Jan 5, 2021",12/22/2020,01/05/2021,NaN,NaN,NaN,NaN,NaN,NaN,united states,NaN
3,Took Prescription Medication for Mental Health...,By Age,United States,50 - 59 years,-1,1,"Mar 30 - Apr 13, 2021",03/30/2021,04/13/2021,NaN,NaN,NaN,NaN,NaN,NaN,united states,NaN
4,Took Prescription Medication for Mental Health...,By Age,United States,60 - 69 years,-1,1,"Mar 30 - Apr 13, 2021",03/30/2021,04/13/2021,NaN,NaN,NaN,NaN,NaN,NaN,united states,NaN


# Ejercicio breve

1. Calcula el promedio de `value` para cada combinación de `state` y `group`.
2. Ordena el resultado de mayor a menor promedio.
3. Combina ese resultado con la tabla `regiones` usando un `left join`.
4. Identifica qué estados no encontraron región.


In [ ]:
# Tu código aquí
